German → English translation using a Transformer encoder-decoder implemented from scratch in PyTorch.

In [6]:
# run this once if spaCy models are missing
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 106.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 120.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
# Mini Transformer from Scratch for German -> English Translation
# Fixed version: positional encoding, masks, device handling, inference, target PAD loss.

import gzip
import math
import random
import time
from collections import Counter
from typing import Callable, Dict, List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm



In [ ]:
# -----------------------------
# 1. Reproducibility + Device


In [2]:
# -----------------------------
SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Using device: cuda


In [3]:
# -----------------------------
# 2. Optional spaCy tokenizers


In [4]:
# -----------------------------
def load_spacy_model(model_name: str):
    try:
        import spacy
        return spacy.load(model_name)
    except Exception as e:
        print(f"Could not load {model_name}. Falling back to simple whitespace tokenizer.")
        print("Install with:")
        print(f"python -m spacy download {model_name}")
        return None

spacy_de = load_spacy_model("de_core_news_sm")
spacy_en = load_spacy_model("en_core_web_sm")

def tokenize_de(text: str) -> List[str]:
    if spacy_de is not None:
        return [token.text.lower() for token in spacy_de.tokenizer(text)]
    return text.lower().strip().split()

def tokenize_en(text: str) -> List[str]:
    if spacy_en is not None:
        return [token.text.lower() for token in spacy_en.tokenizer(text)]
    return text.lower().strip().split()



Could not load de_core_news_sm. Falling back to simple whitespace tokenizer.
Install with:
python -m spacy download de_core_news_sm


In [5]:
# -----------------------------
# 3. Dataset


In [ ]:
# -----------------------------
class Multi30kDataset(Dataset):
    def __init__(
        self,
        src_file: str,
        trg_file: str,
        src_transform: Callable[[str], List[str]] = None,
        trg_transform: Callable[[str], List[str]] = None,
    ):
        self.src_data = self.load_data(src_file)
        self.trg_data = self.load_data(trg_file)
        assert len(self.src_data) == len(self.trg_data), "Source and target files must have same length."
        self.src_transform = src_transform
        self.trg_transform = trg_transform

    def load_data(self, file_path: str) -> List[str]:
        with gzip.open(file_path, "rt", encoding="utf-8") as f:
            return f.readlines()

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        src_sentence = self.src_data[idx].strip()
        trg_sentence = self.trg_data[idx].strip()

        if self.src_transform:
            src_sentence = self.src_transform(src_sentence)
        if self.trg_transform:
            trg_sentence = self.trg_transform(trg_sentence)

        return {"src": src_sentence, "trg": trg_sentence}

In [ ]:
# Update these paths according to your environment.
train_de_path = "/content/data/train.de.gz"
train_en_path = "/content/data/train.en.gz"
val_de_path = "/content/data/val.de.gz"
val_en_path = "/content/data/val.en.gz"
test_de_path = "/content/data/test_2016_flickr.de.gz"
test_en_path = "/content/data/test_2016_flickr.en.gz"

In [ ]:
# -----------------------------
# 4. Vocabulary


In [ ]:
# -----------------------------
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

def create_vocab(tokenized_sentences: List[List[str]], special_tokens: List[str], min_freq: int = 1) -> Dict[str, int]:
    vocab = {token: idx for idx, token in enumerate(special_tokens)}
    counter = Counter(token for sent in tokenized_sentences for token in sent)
    for token, freq in counter.items():
        if freq >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab

def build_datasets_and_vocabs():
    train_data = Multi30kDataset(train_de_path, train_en_path, tokenize_de, tokenize_en)
    val_data = Multi30kDataset(val_de_path, val_en_path, tokenize_de, tokenize_en)
    test_data = Multi30kDataset(test_de_path, test_en_path, tokenize_de, tokenize_en)

    train_de_tokenized = [tokenize_de(sentence.strip()) for sentence in train_data.src_data]
    train_en_tokenized = [tokenize_en(sentence.strip()) for sentence in train_data.trg_data]

    src_vocab = create_vocab(train_de_tokenized, SPECIAL_TOKENS)
    trg_vocab = create_vocab(train_en_tokenized, SPECIAL_TOKENS)

    return train_data, val_data, test_data, src_vocab, trg_vocab



In [ ]:
# -----------------------------
# 5. Transformer Components


In [ ]:
# -----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Shape: [1, max_len, d_model], so it broadcasts over batch dimension.
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [batch_size, seq_len, d_model]
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Q: [B, H, Q_len, d_k]
        # K: [B, H, K_len, d_k]
        # V: [B, H, K_len, d_k]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            # mask should broadcast to [B, H, Q_len, K_len]
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_probs = torch.softmax(scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        output = torch.matmul(attn_probs, V)
        return output

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(output)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        attn_output = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, trg_mask):
        self_attn_output = self.self_attn(x, x, x, trg_mask)
        x = self.norm1(x + self.dropout(self_attn_output))

        cross_attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        trg_vocab_size: int,
        src_pad_idx: int,
        trg_pad_idx: int,
        d_model: int = 256,
        num_heads: int = 8,
        num_layers: int = 3,
        d_ff: int = 512,
        max_seq_length: int = 100,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.d_model = d_model
        self.scale = math.sqrt(d_model)

        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        self.dropout = nn.Dropout(dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def make_src_mask(self, src):
        # src: [B, src_len]
        return (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)  # [B, 1, 1, src_len]

    def make_trg_mask(self, trg):
        # trg: [B, trg_len]
        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(3)  # [B, 1, trg_len, 1]
        trg_len = trg.size(1)
        causal_mask = torch.tril(torch.ones((trg_len, trg_len), device=trg.device)).bool()  # [trg_len, trg_len]
        return trg_pad_mask & causal_mask  # [B, 1, trg_len, trg_len]

    def encode(self, src, src_mask):
        src_embedded = self.dropout(
            self.positional_encoding(self.encoder_embedding(src) * self.scale)
        )
        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)
        return enc_output

    def decode(self, trg, enc_output, src_mask, trg_mask):
        trg_embedded = self.dropout(
            self.positional_encoding(self.decoder_embedding(trg) * self.scale)
        )
        dec_output = trg_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, trg_mask)
        return dec_output

    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(trg, enc_output, src_mask, trg_mask)
        return self.fc_out(dec_output)



In [ ]:
# -----------------------------
# 6. DataLoader + Training Utils


In [ ]:
# -----------------------------
def make_collate_fn(src_vocab: Dict[str, int], trg_vocab: Dict[str, int]):
    def collate_fn(batch):
        src_batch, trg_batch = [], []
        for sample in batch:
            src_ids = [src_vocab.get(tok, src_vocab[UNK_TOKEN]) for tok in [SOS_TOKEN] + sample["src"] + [EOS_TOKEN]]
            trg_ids = [trg_vocab.get(tok, trg_vocab[UNK_TOKEN]) for tok in [SOS_TOKEN] + sample["trg"] + [EOS_TOKEN]]
            src_batch.append(torch.tensor(src_ids, dtype=torch.long))
            trg_batch.append(torch.tensor(trg_ids, dtype=torch.long))

        src_batch = pad_sequence(src_batch, padding_value=src_vocab[PAD_TOKEN], batch_first=True)
        trg_batch = pad_sequence(trg_batch, padding_value=trg_vocab[PAD_TOKEN], batch_first=True)
        return src_batch, trg_batch
    return collate_fn

def train_epoch(model, iterator, optimizer, criterion, clip: float):
    model.train()
    epoch_loss = 0.0

    for src, trg in tqdm(iterator, desc="Training", leave=False):
        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        # Decoder input excludes final token.
        output = model(src, trg[:, :-1])

        # Target excludes first <sos> token.
        output_dim = output.shape[-1]
        output = output.contiguous().view(-1, output_dim)
        target = trg[:, 1:].contiguous().view(-1)

        loss = criterion(output, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0.0

    with torch.no_grad():
        for src, trg in tqdm(iterator, desc="Evaluating", leave=False):
            src = src.to(device)
            trg = trg.to(device)

            output = model(src, trg[:, :-1])
            output_dim = output.shape[-1]
            output = output.contiguous().view(-1, output_dim)
            target = trg[:, 1:].contiguous().view(-1)

            loss = criterion(output, target)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    mins = int(elapsed / 60)
    secs = int(elapsed - (mins * 60))
    return mins, secs



In [ ]:
# -----------------------------
# 7. Inference / Translation


In [ ]:
# -----------------------------
def translate_sentence(
    sentence: str,
    src_vocab: Dict[str, int],
    trg_vocab: Dict[str, int],
    model: Transformer,
    max_len: int = 50,
) -> List[str]:
    model.eval()
    idx_to_trg = {idx: token for token, idx in trg_vocab.items()}

    tokens = [SOS_TOKEN] + tokenize_de(sentence) + [EOS_TOKEN]
    src_indexes = [src_vocab.get(token, src_vocab[UNK_TOKEN]) for token in tokens]
    src_tensor = torch.tensor(src_indexes, dtype=torch.long, device=device).unsqueeze(0)

    src_mask = model.make_src_mask(src_tensor)

    with torch.no_grad():
        enc_output = model.encode(src_tensor, src_mask)

    trg_indexes = [trg_vocab[SOS_TOKEN]]

    for _ in range(max_len):
        trg_tensor = torch.tensor(trg_indexes, dtype=torch.long, device=device).unsqueeze(0)
        trg_mask = model.make_trg_mask(trg_tensor)

        with torch.no_grad():
            dec_output = model.decode(trg_tensor, enc_output, src_mask, trg_mask)
            output = model.fc_out(dec_output)

        pred_token = output.argmax(dim=-1)[:, -1].item()
        trg_indexes.append(pred_token)

        if pred_token == trg_vocab[EOS_TOKEN]:
            break

    trg_tokens = [idx_to_trg.get(idx, UNK_TOKEN) for idx in trg_indexes]
    return trg_tokens[1:-1]



In [ ]:
# -----------------------------
# 8. Main Training Script


In [ ]:

train_data, val_data, test_data, SRC_VOCAB, TRG_VOCAB = build_datasets_and_vocabs()

print("Source vocab size:", len(SRC_VOCAB))
print("Target vocab size:", len(TRG_VOCAB))

# For teaching/demo. Increase these for a stronger model.
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 3
D_FF = 512
MAX_SEQ_LENGTH = 100
DROPOUT = 0.1

model = Transformer(
    src_vocab_size=len(SRC_VOCAB),
    trg_vocab_size=len(TRG_VOCAB),
    src_pad_idx=SRC_VOCAB[PAD_TOKEN],
    trg_pad_idx=TRG_VOCAB[PAD_TOKEN],
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_seq_length=MAX_SEQ_LENGTH,
    dropout=DROPOUT,
).to(device)

print(f"The model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters")

optimizer = optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=TRG_VOCAB[PAD_TOKEN])

BATCH_SIZE = 32
N_EPOCHS = 1
CLIP = 1.0

collate_fn = make_collate_fn(SRC_VOCAB, TRG_VOCAB)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

best_valid_loss = float("inf")

for epoch in range(N_EPOCHS):
    start = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, val_loader, criterion)
    end = time.time()

    mins, secs = epoch_time(start, end)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "transformer-translation-model.pt")

    print(f"Epoch: {epoch + 1:02} | Time: {mins}m {secs}s")
    print(f"\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}")
    print(f"\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}")

model.load_state_dict(torch.load("transformer-translation-model.pt", map_location=device))

print("\nExample translations:\n")
for example_idx in range(3):
    src = test_data[example_idx]["src"]
    trg = test_data[example_idx]["trg"]
    pred = translate_sentence(" ".join(src), SRC_VOCAB, TRG_VOCAB, model)

    print("Source:   ", " ".join(src))
    print("Target:   ", " ".join(trg))
    print("Predicted:", " ".join(pred))
    print()


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/train.de.gz'